# Module 00: Prerequisites Verification

**Companion notebook for**: `00-prerequisites.html`

## Overview

This notebook is your **pre-flight checklist** before the CareerAlign GenAI Course begins. Run each cell top-to-bottom. Green checks mean you are ready; any failures tell you exactly what to fix.

## Learning Objectives
- Verify your Python version and environment are correctly configured
- Install and import all core course packages
- Set up API keys securely using environment variables
- Confirm basic Python fluency (type hints, list comprehensions, async/await, decorators)
- Check hardware capabilities (GPU availability)
- Make a "Hello World" LLM API call to confirm end-to-end connectivity

## Prerequisites
- Python 3.11+ installed
- An OpenAI API key (store in a `.env` file as `OPENAI_API_KEY=sk-...`)

---
## Section 1: Python Version Check

The course requires **Python 3.11 or newer**. Some libraries rely on modern type annotation syntax (`X | Y` unions, `TypedDict`, etc.) that is only available in 3.11+.

In [ ]:
import sys

major, minor, micro = sys.version_info[:3]
print(f"Python version: {major}.{minor}.{micro}")
print(f"Executable:     {sys.executable}")
print(f"Platform:       {sys.platform}")

assert major == 3 and minor >= 11, (
    f"Python 3.11+ required, but you have {major}.{minor}.{micro}. "
    f"Please install a newer version from https://python.org"
)
print("\nPython version check PASSED")

---
## Section 2: Install Core Packages

Run the cell below to install all packages needed throughout the course. If you are working inside a virtual environment (recommended), these will be installed there. The `-q` flag keeps the output quiet.

In [ ]:
# LLM providers & utilities
%pip install -q openai anthropic google-generativeai litellm tiktoken

# Orchestration frameworks
%pip install -q langchain langchain-openai langchain-anthropic langchain-community

# Vector databases
%pip install -q chromadb qdrant-client

# Data science essentials
%pip install -q numpy pandas matplotlib scikit-learn

# Web framework & HTTP
%pip install -q fastapi uvicorn httpx

# Environment & config
%pip install -q python-dotenv pydantic

# AWS (for later modules)
%pip install -q boto3

# Evaluation
%pip install -q ragas

print("\nAll packages installed successfully.")

---
## Section 3: Import Verification

This cell imports every key library used in the course. If any import fails, the error message will tell you exactly which package is missing.

In [ ]:
import importlib

# (module_name, display_name)
REQUIRED_PACKAGES = [
    ("openai", "OpenAI SDK"),
    ("anthropic", "Anthropic SDK"),
    ("langchain", "LangChain"),
    ("langchain_openai", "LangChain-OpenAI"),
    ("chromadb", "ChromaDB"),
    ("numpy", "NumPy"),
    ("pandas", "Pandas"),
    ("matplotlib", "Matplotlib"),
    ("sklearn", "scikit-learn"),
    ("fastapi", "FastAPI"),
    ("httpx", "httpx"),
    ("dotenv", "python-dotenv"),
    ("pydantic", "Pydantic"),
    ("tiktoken", "tiktoken"),
    ("boto3", "boto3 (AWS)"),
]

passed = 0
failed = 0

for module_name, display_name in REQUIRED_PACKAGES:
    try:
        mod = importlib.import_module(module_name)
        version = getattr(mod, "__version__", "(no version attr)")
        print(f"  [OK]   {display_name:25s} {version}")
        passed += 1
    except ImportError as e:
        print(f"  [FAIL] {display_name:25s} -- {e}")
        failed += 1

print(f"\nResults: {passed} passed, {failed} failed out of {len(REQUIRED_PACKAGES)}")
if failed == 0:
    print("All imports PASSED")
else:
    print("Fix the failing imports above before proceeding.")

---
## Section 4: API Key Setup & Connection Test

API keys are **secrets** that must never appear in source code or git history. The standard pattern is:

1. Create a `.env` file in your project root (add it to `.gitignore`)
2. Store keys as `KEY=value` lines
3. Load them with `python-dotenv` at runtime
4. Access via `os.environ["KEY"]`

Example `.env` file:
```
OPENAI_API_KEY=sk-proj-abc123...
ANTHROPIC_API_KEY=sk-ant-abc123...
```

In [ ]:
import os
from dotenv import load_dotenv

# Load .env file from the current directory or parent directories
load_dotenv()

# Check which API keys are available
API_KEYS = [
    "OPENAI_API_KEY",
    "ANTHROPIC_API_KEY",
]

for key_name in API_KEYS:
    value = os.environ.get(key_name, "")
    if value:
        # Show only the first 8 and last 4 characters for verification
        masked = value[:8] + "..." + value[-4:]
        print(f"  [OK]   {key_name} = {masked}")
    else:
        print(f"  [MISS] {key_name} -- not set")

# The course minimally requires an OpenAI key
assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY is not set. Create a .env file with your key "
    "or run: export OPENAI_API_KEY=sk-..."
)
print("\nOpenAI API key found. Ready for LLM calls.")

---
## Section 5: Basic Python Skills Check

The course assumes you are **functionally fluent** in intermediate Python. The cells below demonstrate the exact patterns you will encounter from Day 1. Read each cell and make sure nothing feels unfamiliar.

### 5a: Type Hints

Every function in the course uses type hints. They make code readable and enable IDE autocompletion.

In [ ]:
from typing import Optional


def format_prompt(
    user_query: str,
    system_message: str = "You are a helpful assistant.",
    max_tokens: int = 1024,
    temperature: float | None = None,
) -> dict:
    """Build a chat-completion request payload with type hints."""
    payload: dict = {
        "model": "gpt-4o-mini",
        "max_tokens": max_tokens,
        "messages": [
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_query},
        ],
    }
    if temperature is not None:
        payload["temperature"] = temperature
    return payload


# Demo
payload = format_prompt("What is RAG?", temperature=0.7)
print("Request payload:")
import json
print(json.dumps(payload, indent=2))

### 5b: List Comprehensions & Dictionary Comprehensions

These are used constantly for transforming data -- parsing API responses, building message lists, filtering results.

In [ ]:
# Simulated LLM API response with multiple choices
raw_responses = [
    {"index": 0, "text": "RAG stands for Retrieval-Augmented Generation.", "score": 0.95},
    {"index": 1, "text": "RAG is a framework for grounding LLMs.", "score": 0.88},
    {"index": 2, "text": "I don't know.", "score": 0.12},
]

# List comprehension: extract only high-quality responses
good_responses = [r["text"] for r in raw_responses if r["score"] > 0.5]
print("High-quality responses:", good_responses)

# Dict comprehension: build a lookup from index -> text
response_lookup = {r["index"]: r["text"] for r in raw_responses}
print("\nLookup dict:", response_lookup)

# Nested comprehension: flatten a list of message lists
conversations = [
    [{"role": "user", "content": "Hi"}],
    [{"role": "user", "content": "What is RAG?"}, {"role": "assistant", "content": "RAG is..."}],
]
all_messages = [msg for convo in conversations for msg in convo]
print(f"\nFlattened: {len(all_messages)} messages from {len(conversations)} conversations")

### 5c: Decorators

Decorators appear everywhere in the course: FastAPI routes (`@app.post`), LangChain tools (`@tool`), caching, retry logic. A decorator is syntactic sugar for wrapping a function in another function.

In [ ]:
import time
from functools import wraps


def timing(func):
    """Decorator that prints how long a function takes to run."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper


@timing
def simulate_api_call(prompt: str) -> str:
    """Simulates a slow API call."""
    time.sleep(0.3)
    return f"Response to: {prompt}"


# The decorator wraps the function transparently
result = simulate_api_call("What is GenAI?")
print(f"Result: {result}")
print(f"Function name preserved: {simulate_api_call.__name__}")

### 5d: Pydantic Models

Pydantic is used throughout the course for structured data: API request/response schemas, LLM output parsing, pipeline configuration. It combines class syntax with automatic validation.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional


class DocumentChunk(BaseModel):
    """A chunk of text with metadata -- used in RAG pipelines."""
    content: str = Field(..., description="The text content of this chunk")
    source: str = Field(..., description="Source file or URL")
    chunk_index: int = Field(0, ge=0)
    embedding: Optional[list[float]] = None
    metadata: dict = Field(default_factory=dict)


# Create a chunk -- Pydantic validates types automatically
chunk = DocumentChunk(
    content="Transformers use self-attention to process sequences in parallel.",
    source="module-01.pdf",
    chunk_index=3,
    metadata={"page": 12, "section": "Architecture"},
)

print("Chunk object:", chunk)
print("\nAs dict:", chunk.model_dump())
print("\nAs JSON:", chunk.model_dump_json(indent=2))

# Validation in action -- this will raise a clear error
try:
    bad_chunk = DocumentChunk(content="test", source="test", chunk_index=-1)
except Exception as e:
    print(f"\nValidation error (expected): {e}")

### 5e: Async / Await

LLM API calls are I/O-bound (you wait for a network response). Async Python lets you handle many concurrent requests efficiently. FastAPI is async by default, and most LLM SDKs expose async clients.

In [ ]:
import asyncio


async def fetch_completion(prompt: str, delay: float = 0.2) -> str:
    """Simulate an async LLM API call."""
    await asyncio.sleep(delay)  # Simulates network latency
    return f"Response to: {prompt}"


async def run_concurrent_calls():
    """Run multiple 'API calls' concurrently instead of sequentially."""
    prompts = [
        "What is RAG?",
        "Explain embeddings.",
        "What is a vector database?",
        "Define prompt engineering.",
    ]

    # Sequential: total time = sum of all delays
    start = time.perf_counter()
    sequential_results = []
    for p in prompts:
        sequential_results.append(await fetch_completion(p))
    seq_time = time.perf_counter() - start

    # Concurrent: total time = max delay (all run simultaneously)
    start = time.perf_counter()
    concurrent_results = await asyncio.gather(
        *[fetch_completion(p) for p in prompts]
    )
    conc_time = time.perf_counter() - start

    print(f"Sequential: {seq_time:.3f}s for {len(prompts)} calls")
    print(f"Concurrent: {conc_time:.3f}s for {len(prompts)} calls")
    print(f"Speedup:    {seq_time / conc_time:.1f}x")
    return concurrent_results


# In Jupyter, we can await directly at the top level
results = await run_concurrent_calls()

### 5f: Context Managers

Context managers (`with` statements) guarantee that resources are properly cleaned up -- database connections closed, file handles released, HTTP sessions torn down -- even when exceptions occur.

In [ ]:
from contextlib import contextmanager


@contextmanager
def managed_resource(name: str):
    """Demonstrates the context manager pattern used throughout the course."""
    print(f"  [setup]    Acquiring {name}...")
    resource = {"name": name, "active": True}
    try:
        yield resource  # Hand control to the `with` block
    except Exception as e:
        print(f"  [error]    Error in {name}: {e}")
        raise
    finally:
        resource["active"] = False
        print(f"  [teardown] Released {name}")


# Normal usage
print("Normal flow:")
with managed_resource("DB Connection") as conn:
    print(f"  [use]      Using {conn['name']}, active={conn['active']}")

print(f"  [after]    active={conn['active']}")

# Error handling -- teardown still runs
print("\nError flow:")
try:
    with managed_resource("HTTP Session") as session:
        raise RuntimeError("Connection timeout!")
except RuntimeError:
    print("  [caught]   Exception handled, but resource was still cleaned up")

---
## Section 6: NumPy & Pandas Quick Check

NumPy arrays represent **embedding vectors** (the numeric form of text used in vector search). Pandas DataFrames are used for evaluation results, dataset processing, and exploratory analysis.

In [ ]:
import numpy as np
import pandas as pd

# --- NumPy: Embedding vectors and cosine similarity ---
# In RAG, text chunks are converted to vectors; similarity search
# finds the closest vectors to a query vector.

emb_a = np.array([0.2, 0.8, -0.3, 0.5])   # "What is RAG?"
emb_b = np.array([0.1, 0.9, -0.2, 0.4])   # "Explain RAG"   (similar)
emb_c = np.array([-0.7, 0.1, 0.6, -0.3])  # "Cook a steak"  (unrelated)

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("Cosine similarity (the core metric behind vector search):")
print(f"  'What is RAG?' vs 'Explain RAG':  {cosine_similarity(emb_a, emb_b):.4f}  (high = similar)")
print(f"  'What is RAG?' vs 'Cook a steak': {cosine_similarity(emb_a, emb_c):.4f}  (low = unrelated)")

# --- Pandas: Evaluation results ---
eval_results = pd.DataFrame({
    "question": ["What is RAG?", "How does chunking work?", "Explain embeddings"],
    "faithfulness": [0.92, 0.78, 0.95],
    "relevance": [0.88, 0.91, 0.85],
    "latency_ms": [320, 450, 280],
})

print("\nRAG Evaluation Results:")
print(eval_results.to_string(index=False))

# Filter for low-quality answers
poor = eval_results[eval_results["faithfulness"] < 0.8]
print(f"\nQuestions below faithfulness threshold: {len(poor)}")
print(f"Mean latency: {eval_results['latency_ms'].mean():.0f}ms")

---
## Section 7: GPU / Hardware Check

Most course work uses cloud-hosted LLMs via API (no local GPU required). However, some modules (fine-tuning, local model hosting) benefit from GPU access. This cell checks what is available.

In [ ]:
import platform
import shutil

print("=== System Information ===")
print(f"  OS:           {platform.system()} {platform.release()}")
print(f"  Machine:      {platform.machine()}")
print(f"  Processor:    {platform.processor() or 'N/A'}")

# Check for NVIDIA GPU (PyTorch / CUDA)
print("\n=== GPU Check ===")
try:
    import torch
    if torch.cuda.is_available():
        print(f"  CUDA available: Yes")
        print(f"  GPU:           {torch.cuda.get_device_name(0)}")
        print(f"  VRAM:          {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print(f"  Apple MPS available: Yes (Metal Performance Shaders)")
        print(f"  Suitable for local model inference with MLX or MPS backend.")
    else:
        print(f"  No GPU detected. CPU only.")
        print(f"  This is fine -- most course work uses cloud LLM APIs.")
except ImportError:
    print("  PyTorch not installed -- skipping GPU check.")
    print("  This is fine for API-based modules. Install torch if needed for fine-tuning.")

# Check disk space
total, used, free = shutil.disk_usage("/")
print(f"\n=== Disk Space ===")
print(f"  Free: {free / (1 << 30):.1f} GB")
if free / (1 << 30) < 10:
    print("  WARNING: Less than 10 GB free. Some modules download large models/datasets.")

---
## Section 8: JSON & REST API Fundamentals

Every LLM interaction is JSON over HTTP. This cell demonstrates parsing a typical API response -- the exact pattern you will use throughout the course.

In [ ]:
import json

# A typical LLM API response (what you get back from OpenAI / Anthropic)
response_json = """
{
  "id": "chatcmpl-abc123",
  "object": "chat.completion",
  "model": "gpt-4o-mini",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "RAG stands for Retrieval-Augmented Generation."
      },
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 14,
    "completion_tokens": 9,
    "total_tokens": 23
  }
}
"""

# Parse and navigate the nested structure
data = json.loads(response_json)

# This is the pattern you will write hundreds of times:
answer = data["choices"][0]["message"]["content"]
tokens = data["usage"]["total_tokens"]
model = data["model"]

print(f"Model:   {model}")
print(f"Answer:  {answer}")
print(f"Tokens:  {tokens}")

# Building the messages list (the input format for chat APIs)
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is RAG?"},
    {"role": "assistant", "content": "RAG stands for Retrieval-Augmented Generation."},
    {"role": "user", "content": "How does it work?"},
]
print(f"\nConversation has {len(messages)} messages")
print(f"Serialized size: {len(json.dumps(messages))} bytes")

---
## Section 9: Hello World LLM API Call

The final check: make a real API call to OpenAI. If this cell runs successfully, your environment is fully ready for the course.

In [ ]:
from openai import OpenAI

# The client reads OPENAI_API_KEY from the environment automatically
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

print("Sending request to OpenAI...")

response = client.chat.completions.create(
    model="gpt-4o-mini",
    max_tokens=100,
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Reply concisely."},
        {"role": "user", "content": "In one sentence, what is Retrieval-Augmented Generation (RAG)?"}
    ],
)

# Extract the response
answer = response.choices[0].message.content
usage = response.usage

print(f"\nModel:    {response.model}")
print(f"Answer:   {answer}")
print(f"Tokens:   {usage.prompt_tokens} in + {usage.completion_tokens} out = {usage.total_tokens} total")
print(f"\nHello World LLM call PASSED")

---
## Section 10: Vector Database Quick Test

ChromaDB is the primary vector database used in the RAG modules. This cell verifies it works by creating a small in-memory collection and running a similarity search.

In [ ]:
import chromadb

# Create an ephemeral (in-memory) client -- no files written to disk
chroma_client = chromadb.Client()

# Create a collection (like a table in a database)
collection = chroma_client.create_collection(name="prereq_test")

# Add documents -- ChromaDB generates embeddings automatically
collection.add(
    documents=[
        "RAG combines retrieval with generation for grounded answers.",
        "Fine-tuning adjusts model weights on a custom dataset.",
        "Prompt engineering designs inputs to steer LLM behavior.",
        "Vector databases store embeddings for similarity search.",
    ],
    ids=["doc-1", "doc-2", "doc-3", "doc-4"],
)

# Query -- find the most relevant document
results = collection.query(
    query_texts=["How do I ground my LLM with external data?"],
    n_results=2,
)

print("Query: 'How do I ground my LLM with external data?'\n")
print("Top results:")
for doc, dist in zip(results["documents"][0], results["distances"][0]):
    print(f"  [{dist:.4f}] {doc}")

print(f"\nChromaDB check PASSED ({collection.count()} documents indexed)")

---
## Final Checklist

If you made it here with all cells passing, you are ready for the course.

| Check | Status |
|---|---|
| Python 3.11+ | Run Section 1 |
| Core packages installed | Run Section 2-3 |
| API keys configured | Run Section 4 |
| Python fundamentals comfortable | Read Section 5 |
| NumPy & Pandas basics | Run Section 6 |
| Hardware checked | Run Section 7 |
| JSON parsing understood | Run Section 8 |
| LLM API call works | Run Section 9 |
| Vector DB works | Run Section 10 |

**Next step**: Open `01-foundations-of-genai.html` to begin Module 01.